# Kapitola 1: Základní struktura promptu

- [Lekce](#lesson)
- [Cvičení](#exercises)
- [Příkladové hřiště](#example-playground)

## Nastavení

Spusťte následující buňku pro nastavení, abyste načetli svůj API klíč a vytvořili pomocnou funkci `get_completion`.

In [ ]:
```python
%pip install anthropic --quiet

# Importujte modul hints z balíčku utils
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Importujte vestavěnou knihovnu regulárních výrazů v Pythonu
import re
from anthropic import AnthropicBedrock

%store -r MODEL_NAME
%store -r AWS_REGION

client = AnthropicBedrock(aws_region=AWS_REGION)

def get_completion(prompt, system=''):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": prompt}
        ],
        system=system
    )
    return message.content[0].text
```

## Lekce

Anthropic nabízí dvě API, starší [Text Completions API](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-anthropic-claude-text-completion.html) a aktuální [Messages API](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-anthropic-claude-messages.html). Pro tento tutoriál budeme používat výhradně Messages API.

Minimálně vyžaduje volání Claude pomocí Messages API následující parametry:
- `model`: [API model name](https://docs.aws.amazon.com/bedrock/latest/userguide/model-ids.html#model-ids-arns) modelu, který chcete volat

- `max_tokens`: maximální počet tokenů, které se mají vygenerovat před zastavením. Všimněte si, že Claude může zastavit před dosažením tohoto maxima. Tento parametr pouze určuje absolutní maximální počet tokenů k vygenerování. Dále se jedná o *tvrdé* zastavení, což znamená, že může způsobit, že Claude přestane generovat uprostřed slova nebo věty.

- `messages`: pole vstupních zpráv. Naše modely jsou trénovány k práci na střídavých konverzačních tazích `user` a `assistant`. Při vytváření nové `Message` specifikujete předchozí konverzační tahy pomocí parametru messages a model pak generuje další `Message` v konverzaci.
  - Každá vstupní zpráva musí být objekt s `role` a `content`. Můžete specifikovat jednu zprávu s rolí `user`, nebo můžete zahrnout více zpráv `user` a `assistant` (musí se střídat, pokud ano). První zpráva musí vždy používat roli `user`.

Existují také volitelné parametry, jako například:
- `system`: systémový prompt - více o tom níže.
  
- `temperature`: stupeň variability v odpovědi Claude. Pro tyto lekce a cvičení jsme nastavili `temperature` na 0.

Pro kompletní seznam všech API parametrů navštivte naši [API dokumentaci](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-claude.html).

### Příklady

Podívejme se, jak Claude reaguje na některé správně formátované prompty. Pro každý z následujících bloků spusťte buňku (`shift+enter`) a odpověď Claude se objeví pod blokem.

In [ ]:
```python
# Prompt
PROMPT = "Hi Claude, how are you?"

# Vytiskne odpověď od Claude
print(get_completion(PROMPT))
```

In [ ]:
```python
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

In [ ]:
```python
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

Nyní se podívejme na některé prompty, které neobsahují správné formátování Messages API. Pro tyto špatně formátované prompty vrací Messages API chybu.

Nejprve máme příklad volání Messages API, které postrádá pole `role` a `content` v poli `messages`.

> ⚠️ **Varování:** Kvůli nesprávnému formátování parametru messages v promptu vrátí následující buňka chybu. Toto je očekávané chování.

In [ ]:
```python
# Získat odpověď od Claude
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Vytisknout odpověď od Claude
print(response[0].text)
```

Zde je prompt, který selhává ve střídání mezi rolemi `user` a `assistant`.

> ⚠️ **Varování:** Kvůli nedostatku střídání mezi rolemi `user` a `assistant` vrátí Claude chybovou zprávu. Toto je očekávané chování.

In [ ]:
```python
# Získat odpověď od Claude
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Vytisknout odpověď od Claude
print(response[0].text)
```

`user` a `assistant` zprávy **MUSÍ střídat**, a zprávy **MUSÍ začínat `user` částí**. Můžete mít více párů `user` & `assistant` v promptu (jako byste simulovali konverzaci s více tahy). Můžete také vložit slova do koncové `assistant` zprávy, aby Claude mohl pokračovat tam, kde jste skončili (více o tom v pozdějších kapitolách).

#### Systémové Prompty

Můžete také použít **systémové prompty**. Systémový prompt je způsob, jak **poskytnout kontext, instrukce a pokyny Claudovi** před tím, než mu v "User" části položíte otázku nebo úkol.

Strukturálně existují systémové prompty odděleně od seznamu `user` & `assistant` zpráv, a proto patří do samostatného parametru `system` (podívejte se na strukturu pomocné funkce `get_completion` v sekci [Setup](#setup) v notebooku).

V rámci tohoto tutoriálu, kdykoli bychom mohli využít systémový prompt, jsme vám poskytli pole `system` ve vaší funkci pro dokončení. Pokud nechcete použít systémový prompt, jednoduše nastavte proměnnou `SYSTEM_PROMPT` na prázdný řetězec.

#### Příklad systémového promptu

In [ ]:
```python
# Systémový prompt
SYSTEM_PROMPT = "Vaše odpověď by měla být vždy sérií otázek podporujících kritické myšlení, které posunou konverzaci dál (neposkytujte odpovědi na vaše otázky). Skutečně neodpovídejte na otázku uživatele."

# Prompt
PROMPT = "Proč je obloha modrá?"

# Vytisknout odpověď od Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))
```

Proč používat systémový prompt? **Dobře napsaný systémový prompt může zlepšit výkon Claude** různými způsoby, například zvýšením schopnosti Claude dodržovat pravidla a instrukce. Pro více informací navštivte naši dokumentaci o [jak používat systémové prompty](https://docs.anthropic.com/claude/docs/how-to-use-system-prompts) s Claude.

Nyní se ponoříme do několika cvičení. Pokud byste chtěli experimentovat s prompty lekce, aniž byste měnili jakýkoli obsah výše, přejděte až na konec poznámkového bloku lekce a navštivte [**Example Playground**](#example-playground).

## Cvičení
- [Cvičení 1.1 - Počítání do tří](#exercise-11---counting-to-three)
- [Cvičení 1.2 - System Prompt](#exercise-12---system-prompt)

### Cvičení 1.1 - Počítání do tří
Použijte správné formátování `user` / `assistant`, upravte níže uvedený `PROMPT`, aby Claude **počítal do tří.** Výstup také ukáže, zda je vaše řešení správné.

In [ ]:
```python
# Prompt - toto je jediné pole, které byste měli změnit
PROMPT = "[Replace this text]"

# Získání odpovědi od Claude
response = get_completion(PROMPT)

# Funkce pro hodnocení správnosti cvičení
def grade_exercise(text):
    pattern = re.compile(r'^(?=.*1)(?=.*2)(?=.*3).*$', re.DOTALL)
    return bool(pattern.match(text))

# Výpis odpovědi od Claude a odpovídající hodnocení
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("Toto cvičení bylo správně vyřešeno:", grade_exercise(response))
```

❓ Pokud chcete nápovědu, spusťte buňku níže!

In [ ]:
print(hints.exercise_1_1_hint)

### Cvičení 1.2 - System Prompt

Upravte `SYSTEM_PROMPT`, aby Claude odpovídal jako tříleté dítě.

In [ ]:
```python
# Systémový prompt - toto je jediné pole, které byste měli změnit
SYSTEM_PROMPT = "[Replace this text]"

# Prompt
PROMPT = "How big is the sky?"

# Získání odpovědi od Claude
response = get_completion(PROMPT, SYSTEM_PROMPT)

# Funkce pro hodnocení správnosti cvičení
def grade_exercise(text):
    return bool(re.search(r"giggles", text) or re.search(r"soo", text))

# Tisk odpovědi od Claude a odpovídající hodnocení
print(response)
print("\n--------------------------- HODNOCENÍ ---------------------------")
print("Toto cvičení bylo správně vyřešeno:", grade_exercise(response))
```

❓ Pokud chcete nápovědu, spusťte buňku níže!

In [ ]:
print(hints.exercise_1_2_hint)

### Gratulujeme!

Pokud jste vyřešili všechny úkoly až do tohoto bodu, jste připraveni přejít k další kapitole. Šťastné promptování!

---

## Příklad hřiště

Toto je prostor, kde můžete volně experimentovat s příklady promptů uvedenými v této lekci a upravovat prompty, abyste viděli, jak to může ovlivnit odpovědi Claude.

In [ ]:
```python
# Prompt
PROMPT = "Hi Claude, how are you?"

# Vytiskne odpověď od Claude
print(get_completion(PROMPT))
```

In [ ]:
```python
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

In [ ]:
```python
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

In [ ]:
```python
# Získat odpověď od Claude
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Vytisknout odpověď od Claude
print(response[0].text)
```

In [ ]:
```python
# Získat odpověď od Claude
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Vytisknout odpověď od Claude
print(response[0].text)
```

In [ ]:
```python
# Systémový prompt
SYSTEM_PROMPT = "Vaše odpověď by měla být vždy sérií otázek podporujících kritické myšlení, které posunou konverzaci dále (neposkytujte odpovědi na vaše otázky). Skutečně neodpovídejte na otázku uživatele."

# Prompt
PROMPT = "Proč je obloha modrá?"

# Vytisknout odpověď od Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))
```